# Cell 1 : Install and import

In [2]:
import shutil
import argparse
import importlib.util
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

# Cell 2 : Connect google drive   

In [3]:
# Optional: mount drive if you want save to Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Cell 3 : Create payoff-matrix dataset (.npz)

In [ ]:
# ============================================================
# Cell: Create One Combined Payoff-Matrix Dataset (.npz)
#       3 seeds × 100 games = 300 games
# ============================================================

import os
import numpy as np

# ---------- Configuration ----------
NUM_SEEDS = 3
GAMES_PER_SEED = 50
TOTAL_GAMES = NUM_SEEDS * GAMES_PER_SEED

M = 20
N = 20

DISTRIBUTION = "uniform"  # "normal" or "uniform"
BASE_SEED = 2026

GOOGLEDRIVE = "/content/drive/MyDrive/Research/SLSQP/datasets"
os.makedirs(GOOGLEDRIVE, exist_ok=True)

OUTPUT_FILE = (
    f"input_games_{DISTRIBUTION}_{M}_{N}.npz"
)
OUTPUT_PATH = os.path.join(GOOGLEDRIVE, OUTPUT_FILE)

# Parameters for N(mean, variance)
MEAN = 50.0
VARIANCE = 25.0
STD = np.sqrt(VARIANCE)

# Parameters for U(low, high)
LOW = 0.0
HIGH = 100.0
# ----------------------------------

# Shape:
# payoff_A, payoff_B: (300, 1000, 1000)
payoff_A = np.empty((TOTAL_GAMES, M, N), dtype=np.float32)
payoff_B = np.empty((TOTAL_GAMES, M, N), dtype=np.float32)

# Stores the seed used to generate each game.
seed_ids = np.empty(TOTAL_GAMES, dtype=np.int32)

game_index = 0

for seed_offset in range(NUM_SEEDS):

    current_seed = BASE_SEED + seed_offset
    rng = np.random.default_rng(current_seed)

    print(
        f"Generating seed {seed_offset + 1}/{NUM_SEEDS} "
        f"(seed={current_seed})..."
    )

    for _ in range(GAMES_PER_SEED):

        if DISTRIBUTION == "normal":

            payoff_A[game_index] = rng.normal(
                loc=MEAN,
                scale=STD,
                size=(M, N)
            ).astype(np.float32)

            payoff_B[game_index] = rng.normal(
                loc=MEAN,
                scale=STD,
                size=(M, N)
            ).astype(np.float32)

        elif DISTRIBUTION == "uniform":

            payoff_A[game_index] = rng.uniform(
                low=LOW,
                high=HIGH,
                size=(M, N)
            ).astype(np.float32)

            payoff_B[game_index] = rng.uniform(
                low=LOW,
                high=HIGH,
                size=(M, N)
            ).astype(np.float32)

        else:
            raise ValueError(
                "DISTRIBUTION must be 'normal' or 'uniform'."
            )

        seed_ids[game_index] = current_seed
        game_index += 1

# Save all games into one compressed NPZ file.
np.savez_compressed(
    OUTPUT_PATH,
    payoff_A=payoff_A,
    payoff_B=payoff_B,
    seed_ids=seed_ids,
    distribution=DISTRIBUTION,
    games_per_seed=GAMES_PER_SEED,
    num_seeds=NUM_SEEDS,
)

print("=" * 70)
print(f"Dataset saved to: {OUTPUT_PATH}")
print("payoff_A shape:", payoff_A.shape)
print("payoff_B shape:", payoff_B.shape)
print("seed_ids shape:", seed_ids.shape)
print(f"Total games: {TOTAL_GAMES}")


Generating seed 1/3 (seed=2026)...
Generating seed 2/3 (seed=2027)...
Generating seed 3/3 (seed=2028)...
Dataset saved to: /content/drive/MyDrive/Research/SLSQP/datasets/input_games_uniform_20_20.npz
payoff_A shape: (150, 20, 20)
payoff_B shape: (150, 20, 20)
seed_ids shape: (150,)
Total games: 150


# Cell 4 : Download datasets from google drive

In [4]:
# Copy all files and subfolders from Google Drive to /content
!cp -r /content/drive/MyDrive/Research/SLSQP/datasets/. /content/
print("Download completed")


Download completed


# Cell 5 : Run

In [ ]:
"""Generate general-sum bimatrix games and refine mixed strategies with SLSQP.

Example:
    !python general_sum_refinement_dataset.py --num-games 100 --num-restarts 50

The output .npz contains payoff_A, payoff_B, strategy_p, strategy_q,
R1, R2, epsilon, objective, solver_success, and best_restart.
"""

import argparse
import importlib.util
import multiprocessing as mp
import queue
import subprocess
import sys
import time
import warnings
from pathlib import Path

# The script is self-contained: install missing dependencies automatically.
missing_packages = [name for name in ("numpy", "scipy") if importlib.util.find_spec(name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_packages])

import numpy as np
from scipy.optimize import minimize


class TimeLimitReached(RuntimeError):
    """Raised internally when a per-game time budget is exhausted."""


def time_limit_reached(deadline):
    """Return True when the optional absolute deadline has passed."""
    return deadline is not None and time.perf_counter() >= deadline


def normalize_strategy(x, eps=1e-12):
    """Return a valid probability distribution."""
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, 0.0, None)
    if x.sum() < eps:
        return np.ones_like(x) / len(x)
    return x / x.sum()


def generate_general_sum_game(m, n, distribution, mean, std, low, high, rng):
    """Create two independent M x N payoff matrices."""
    if distribution == "normal":
        payoff_a = rng.normal(mean, std, size=(m, n))
        payoff_b = rng.normal(mean, std, size=(m, n))
    elif distribution == "uniform":
        payoff_a = rng.uniform(low, high, size=(m, n))
        payoff_b = rng.uniform(low, high, size=(m, n))
    else:
        raise ValueError("distribution must be 'normal' or 'uniform'.")
    return payoff_a.astype(np.float64), payoff_b.astype(np.float64)


def compute_regrets(p, q, payoff_a, payoff_b):
    """Compute R1, R2, epsilon=max(R1,R2), and expected utilities."""
    p = normalize_strategy(p)
    q = normalize_strategy(q)

    u1 = float(p @ payoff_a @ q)
    u2 = float(p @ payoff_b @ q)
    r1 = max(0.0, float(np.max(payoff_a @ q) - u1))
    r2 = max(0.0, float(np.max(p @ payoff_b) - u2))
    return {"R1": r1, "R2": r2, "epsilon": max(r1, r2), "u1": u1, "u2": u2}


def refine_general_sum(initial_p, initial_q, payoff_a, payoff_b, maxiter, ftol,
                       deadline=None):
    """Minimize R1 + R2 from an initial pair of mixed strategies."""
    m, n = payoff_a.shape
    if payoff_b.shape != (m, n):
        raise ValueError("payoff_A and payoff_B must have identical shapes.")

    x0 = np.concatenate([normalize_strategy(initial_p), normalize_strategy(initial_q)])

    def objective(x):
        p, q = x[:m], x[m:]
        regrets = compute_regrets(p, q, payoff_a, payoff_b)
        return regrets["R1"] + regrets["R2"]

    constraints = [
        {"type": "eq", "fun": lambda x: np.sum(x[:m]) - 1.0},
        {"type": "eq", "fun": lambda x: np.sum(x[m:]) - 1.0},
    ]
    best_x = x0.copy()
    best_value = objective(best_x)
    timed_out = False

    if time_limit_reached(deadline):
        p = normalize_strategy(best_x[:m])
        q = normalize_strategy(best_x[m:])
        return {"p": p, "q": q, "metrics": compute_regrets(p, q, payoff_a, payoff_b),
                "objective": float(best_value), "success": False, "iterations": 0,
                "timed_out": True}

    def callback(xk):
        nonlocal best_x, best_value
        value = objective(xk)
        if value < best_value:
            best_x, best_value = xk.copy(), value
        if time_limit_reached(deadline):
            raise TimeLimitReached

    try:
        result = minimize(
            objective, x0, method="SLSQP", bounds=[(0.0, 1.0)] * (m + n),
            constraints=constraints, callback=callback,
            options={"maxiter": maxiter, "ftol": ftol, "disp": False},
        )
        candidate_x = result.x
        success = bool(result.success)
        iterations = int(result.nit)
        candidate_value = float(result.fun)
        if candidate_value < best_value:
            best_x, best_value = candidate_x.copy(), candidate_value
    except TimeLimitReached:
        timed_out = True
        success = False
        iterations = 0

    p = normalize_strategy(best_x[:m])
    q = normalize_strategy(best_x[m:])
    return {
        "p": p,
        "q": q,
        "metrics": compute_regrets(p, q, payoff_a, payoff_b),
        "objective": float(best_value),
        "success": success,
        "iterations": iterations,
        "timed_out": timed_out,
    }


def solve_multistart(payoff_a, payoff_b, num_restarts, rng, maxiter, ftol, deadline=None):
    """Use multiple random starts and retain the smallest epsilon solution."""
    m, n = payoff_a.shape
    best = None
    for restart in range(1, num_restarts + 1):
        if time_limit_reached(deadline):
            break
        initial_p = rng.dirichlet(np.ones(m))
        initial_q = rng.dirichlet(np.ones(n))
        candidate = refine_general_sum(initial_p, initial_q, payoff_a, payoff_b, maxiter, ftol, deadline)
        candidate["restart"] = restart
        if best is None or candidate["metrics"]["epsilon"] < best["metrics"]["epsilon"]:
            best = candidate
        if candidate["timed_out"]:
            break
    if best is None:
        p = np.full(payoff_a.shape[0], 1.0 / payoff_a.shape[0])
        q = np.full(payoff_a.shape[1], 1.0 / payoff_a.shape[1])
        metrics = compute_regrets(p, q, payoff_a, payoff_b)
        best = {"p": p, "q": q, "metrics": metrics, "objective": metrics["R1"] + metrics["R2"],
                "success": False, "iterations": 0, "restart": 0, "timed_out": True}
    return best


class GANashGeneralSum:
    """GA equilibrium search with tournament selection and stagnation immigration."""

    def __init__(self, payoff_a, payoff_b, rng, pop_size=150, mutation_rate=0.1,
                 mutation_sigma=0.05, generations=1000, verbose=False):
        self.A = payoff_a
        self.B = payoff_b
        self.m, self.n = payoff_a.shape
        self.rng = rng
        self.pop_size = pop_size
        self.base_mutation_rate = mutation_rate
        self.current_mutation_rate = mutation_rate
        self.base_mutation_sigma = mutation_sigma
        self.current_mutation_sigma = mutation_sigma
        self.generations = generations
        self.verbose = verbose
        self.population = self._initialize_population()

    def _initialize_population(self):
        return self._generate_random_subset(self.pop_size)

    def _generate_random_subset(self, count):
        population = []
        for _ in range(count):
            p = self.rng.random(self.m)
            q = self.rng.random(self.n)
            population.append(np.concatenate([p / p.sum(), q / q.sum()]))
        return np.asarray(population)

    def _get_regrets(self, chromosome):
        p = normalize_strategy(chromosome[:self.m])
        q = normalize_strategy(chromosome[self.m:])
        metrics = compute_regrets(p, q, self.A, self.B)
        return metrics["R1"], metrics["R2"]

    def _fitness(self, chromosome):
        r1, r2 = self._get_regrets(chromosome)
        return 1.0 / (1.0 + r1 + r2)

    def evolve(self, epsilon_threshold=None, deadline=None):
        best_chromosome = None
        best_fitness = -np.inf
        best_regrets = (np.inf, np.inf)
        stagnation_counter = 0
        max_stagnation = 80
        immigration_rate = 0.20

        if self.verbose:
            print("Starting GA evolution with immigration...")

        for generation in range(self.generations):
            if time_limit_reached(deadline):
                break
            fitness_scores = np.asarray([self._fitness(ind) for ind in self.population])
            current_best_idx = np.argmax(fitness_scores)

            if fitness_scores[current_best_idx] > best_fitness:
                best_fitness = fitness_scores[current_best_idx]
                best_chromosome = self.population[current_best_idx].copy()
                best_regrets = self._get_regrets(best_chromosome)
                stagnation_counter = 0
                self.current_mutation_rate = self.base_mutation_rate
                self.current_mutation_sigma = self.base_mutation_sigma

                if epsilon_threshold is not None and max(best_regrets) < epsilon_threshold:
                    if self.verbose:
                        print(f"Convergence reached at generation {generation}.")
                    break
            else:
                stagnation_counter += 1
                if stagnation_counter >= max_stagnation:
                    num_immigrants = max(1, int(self.pop_size * immigration_rate))
                    worst_indices = np.argsort(fitness_scores)[:num_immigrants]
                    self.population[worst_indices] = self._generate_random_subset(num_immigrants)
                    stagnation_counter = 0
                    # Recompute scores because immigration changed the population.
                    fitness_scores = np.asarray([self._fitness(ind) for ind in self.population])
                    if self.verbose:
                        print(f"Stagnation at generation {generation}: injected {num_immigrants} immigrants.")

            new_population = [best_chromosome.copy()]  # Elitism
            while len(new_population) < self.pop_size:
                candidates = self.rng.choice(len(self.population), size=3, replace=False)
                winner = candidates[np.argmax(fitness_scores[candidates])]
                child = self.population[winner].copy()
                if self.rng.random() < self.current_mutation_rate:
                    child += self.rng.normal(0.0, self.current_mutation_sigma, self.m + self.n)
                new_population.append(child)
            self.population = np.asarray(new_population)

            if self.verbose and generation % 100 == 0:
                print(f"Generation {generation}: best epsilon = {max(best_regrets):.6f}")

        p = normalize_strategy(best_chromosome[:self.m])
        q = normalize_strategy(best_chromosome[self.m:])
        return p, q, best_regrets


def genetic_algorithm_initialization(payoff_a, payoff_b, rng, population_size, generations,
                                    mutation_scale, deadline=None):
    """Find a GA initialization using GANashGeneralSum."""
    solver = GANashGeneralSum(
        payoff_a, payoff_b, rng=rng, pop_size=population_size,
        mutation_rate=0.1, mutation_sigma=mutation_scale, generations=generations,
    )
    p, q, _ = solver.evolve(deadline=deadline)
    return p, q


def softmax_np(x, temperature):
    """Numerically stable NumPy softmax."""
    z = x / max(temperature, 1e-12)
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / (exp_z.sum() + 1e-12)


def informed_random_initialization(payoff_a, payoff_b, rng, num_steps=10,
                                  damping=0.3, noise_ratio=0.03):
    """Create one payoff-informed, randomly perturbed feasible strategy pair."""
    m, n = payoff_a.shape
    p = np.full(m, 1.0 / m)
    q = np.full(n, 1.0 / n)
    payoff_scale = np.std(np.concatenate([payoff_a.ravel(), payoff_b.ravel()])) + 1e-12
    temperature = 0.5 * payoff_scale

    for _ in range(num_steps):
        br_p = softmax_np(payoff_a @ q, temperature)
        br_q = softmax_np(p @ payoff_b, temperature)
        p = normalize_strategy((1.0 - damping) * p + damping * br_p)
        q = normalize_strategy((1.0 - damping) * q + damping * br_q)

    p = normalize_strategy((1.0 - noise_ratio) * p + noise_ratio * rng.dirichlet(np.ones(m)))
    q = normalize_strategy((1.0 - noise_ratio) * q + noise_ratio * rng.dirichlet(np.ones(n)))
    return p, q


def smooth_epsilon_warm_start(payoff_a, payoff_b, steps=1000, learning_rate=0.05,
                              beta_schedule=(5.0, 10.0, 15.0, 20.0, 30.0),
                              beta_eps=20.0, eval_interval=10, device=None,
                              verbose=False, deadline=None):
    """Optimize smooth epsilon with temperature annealing and exact checkpoints."""
    if importlib.util.find_spec("torch") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch"])

    import torch
    import torch.nn.functional as F

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif str(device).startswith("cuda") and not torch.cuda.is_available():
        print("Warning: CUDA was requested, but this PyTorch build has no usable CUDA support. Falling back to CPU.")
        device = "cpu"

    a_original = torch.as_tensor(payoff_a, dtype=torch.float32, device=device)
    b_original = torch.as_tensor(payoff_b, dtype=torch.float32, device=device)
    m, n = a_original.shape

    common_mean = 0.5 * (a_original.mean() + b_original.mean())
    common_scale = torch.sqrt(0.5 * (
        a_original.var(unbiased=False) + b_original.var(unbiased=False)
    )).clamp_min(1e-6)
    a = (a_original - common_mean) / common_scale
    b = (b_original - common_mean) / common_scale

    logits_p = torch.zeros(m, dtype=torch.float32, device=device, requires_grad=True)
    logits_q = torch.zeros(n, dtype=torch.float32, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([logits_p, logits_q], lr=learning_rate)

    best_p = np.full(m, 1.0 / m)
    best_q = np.full(n, 1.0 / n)
    best_exact_epsilon = compute_regrets(best_p, best_q, payoff_a, payoff_b)["epsilon"]
    history = []
    steps_per_stage = max(1, int(np.ceil(steps / len(beta_schedule))))

    for step in range(steps):
        if time_limit_reached(deadline):
            break
        stage = min(step // steps_per_stage, len(beta_schedule) - 1)
        beta_br = beta_schedule[stage]
        optimizer.zero_grad()

        p = F.softmax(logits_p, dim=0)
        q = F.softmax(logits_q, dim=0)
        pure_a = a @ q
        pure_b = p @ b
        expected_a = p @ pure_a
        expected_b = pure_b @ q
        smooth_br_a = torch.logsumexp(beta_br * pure_a, dim=0) / beta_br
        smooth_br_b = torch.logsumexp(beta_br * pure_b, dim=0) / beta_br
        smooth_r_a = smooth_br_a - expected_a
        smooth_r_b = smooth_br_b - expected_b
        smooth_epsilon = torch.logsumexp(
            beta_eps * torch.stack([smooth_r_a, smooth_r_b]), dim=0
        ) / beta_eps
        smooth_epsilon.backward()
        torch.nn.utils.clip_grad_norm_([logits_p, logits_q], max_norm=1.0)
        optimizer.step()
        history.append(float(smooth_epsilon.detach().cpu()))

        if step == 0 or (step + 1) % eval_interval == 0 or step == steps - 1:
            with torch.no_grad():
                p_eval = F.softmax(logits_p, dim=0).cpu().numpy()
                q_eval = F.softmax(logits_q, dim=0).cpu().numpy()
            exact_epsilon = compute_regrets(p_eval, q_eval, payoff_a, payoff_b)["epsilon"]
            if exact_epsilon < best_exact_epsilon:
                best_exact_epsilon = exact_epsilon
                best_p, best_q = p_eval.copy(), q_eval.copy()
            if verbose and ((step + 1) % 100 == 0 or step == steps - 1):
                print(
                    f"Step {step + 1:4d}/{steps} | beta_BR={beta_br:.1f} | "
                    f"best exact epsilon={best_exact_epsilon:.6f}"
                )

    return best_p, best_q, history, best_exact_epsilon


def _lemke_howson_worker(payoff_a, payoff_b, dropped_label, result_queue):
    """Run one Lemke--Howson path in a child process for hard timeouts."""
    try:
        import nashpy as nash
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            p, q = nash.Game(payoff_a, payoff_b).lemke_howson(
                initial_dropped_label=int(dropped_label)
            )
        result_queue.put(("ok", p, q))
    except BaseException as error:
        result_queue.put(("error", repr(error)))


def lemke_howson_solve(payoff_a, payoff_b, rng, max_labels=1, epsilon_tol=1e-6,
                       deadline=None):
    """Run Lemke--Howson with a hard, per-game wall-clock timeout."""
    if importlib.util.find_spec("nashpy") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nashpy"])
    try:
        import nashpy as nash
    except ImportError as error:
        raise ImportError(
            "Lemke--Howson requires nashpy. Install it with: !pip install nashpy"
        ) from error

    m, n = payoff_a.shape
    labels = np.arange(m + n)
    rng.shuffle(labels)
    if max_labels > 0:
        labels = labels[:min(max_labels, len(labels))]

    best_result = None
    timed_out = False

    for dropped_label in labels:
        if time_limit_reached(deadline):
            timed_out = True
            break
        result_queue = mp.Queue(maxsize=1)
        worker = mp.Process(
            target=_lemke_howson_worker,
            args=(payoff_a, payoff_b, int(dropped_label), result_queue),
        )
        worker.start()
        remaining = None if deadline is None else max(0.0, deadline - time.perf_counter())
        worker.join(timeout=remaining)

        if worker.is_alive():
            worker.terminate()
            worker.join()
            result_queue.close()
            result_queue.join_thread()
            timed_out = True
            break

        try:
            status, *payload = result_queue.get(timeout=0.2)
            if status != "ok":
                continue
            p, q = payload
            p, q = normalize_strategy(p), normalize_strategy(q)
            metrics = compute_regrets(p, q, payoff_a, payoff_b)
            candidate = {
                "p": p,
                "q": q,
                "metrics": metrics,
                "objective": metrics["R1"] + metrics["R2"],
                "success": metrics["epsilon"] <= epsilon_tol,
                "iterations": 0,
                "restart": int(dropped_label),
                "refinement_applied": False,
                "timed_out": False,
            }
            if best_result is None or metrics["epsilon"] < best_result["metrics"]["epsilon"]:
                best_result = candidate
        except (queue.Empty, Exception):
            continue
        finally:
            result_queue.close()
            result_queue.join_thread()

    if best_result is not None:
        best_result["timed_out"] = timed_out
        return best_result

    # Preserve the output schema if all Lemke--Howson paths fail.
    p = np.full(m, 1.0 / m)
    q = np.full(n, 1.0 / n)
    metrics = compute_regrets(p, q, payoff_a, payoff_b)
    return {
        "p": p,
        "q": q,
        "metrics": metrics,
        "objective": metrics["R1"] + metrics["R2"],
        "success": False,
        "iterations": 0,
        "restart": -1,
        "refinement_applied": False,
        "timed_out": timed_out,
    }


def solve_with_initialization(payoff_a, payoff_b, args, rng):
    """Choose either random multi-start or GA initialization, then refine."""
    deadline = None if args.time_limit <= 0 else time.perf_counter() + args.time_limit
    if args.init_method == "random":
        result = solve_multistart(
            payoff_a, payoff_b, args.num_restarts, rng, args.maxiter, args.ftol, deadline
        )
        result["refinement_applied"] = True
        return result

    if args.init_method == "informed_random":
        best_result = None
        for restart in range(1, args.num_restarts + 1):
            if time_limit_reached(deadline):
                break
            initial_p, initial_q = informed_random_initialization(payoff_a, payoff_b, rng)
            candidate = refine_general_sum(
                initial_p, initial_q, payoff_a, payoff_b, args.maxiter, args.ftol, deadline
            )
            candidate["restart"] = restart
            candidate["refinement_applied"] = True
            if best_result is None or candidate["metrics"]["epsilon"] < best_result["metrics"]["epsilon"]:
                best_result = candidate
            if candidate["timed_out"]:
                break
        if best_result is None:
            m, n = payoff_a.shape
            p, q = np.full(m, 1.0 / m), np.full(n, 1.0 / n)
            metrics = compute_regrets(p, q, payoff_a, payoff_b)
            best_result = {"p": p, "q": q, "metrics": metrics,
                           "objective": metrics["R1"] + metrics["R2"], "success": False,
                           "iterations": 0, "restart": 0, "refinement_applied": True,
                           "timed_out": True}
        return best_result

    if args.init_method == "smooth_slsqp":
        initial_p, initial_q, _, warm_start_epsilon = smooth_epsilon_warm_start(
            payoff_a, payoff_b, steps=args.smooth_steps,
            learning_rate=args.smooth_lr, device=args.smooth_device, deadline=deadline,
        )
        result = refine_general_sum(
            initial_p, initial_q, payoff_a, payoff_b, args.maxiter, args.ftol, deadline
        )
        result["restart"] = 1
        result["refinement_applied"] = True
        result["warm_start_epsilon"] = warm_start_epsilon
        return result

    if args.init_method in {"smooth_only", "smooth_norm_only"}:
        initial_p, initial_q, history, warm_start_epsilon = smooth_epsilon_warm_start(
            payoff_a, payoff_b, steps=args.smooth_steps,
            learning_rate=args.smooth_lr, device=args.smooth_device, deadline=deadline,
        )
        metrics = compute_regrets(initial_p, initial_q, payoff_a, payoff_b)
        return {
            "p": initial_p,
            "q": initial_q,
            "metrics": metrics,
            "objective": metrics["R1"] + metrics["R2"],
            "success": False,
            "iterations": len(history),
            "restart": 1,
            "refinement_applied": False,
            "warm_start_epsilon": warm_start_epsilon,
            "timed_out": time_limit_reached(deadline),
        }


    if args.init_method == "lemke_howson":
        return lemke_howson_solve(
            payoff_a, payoff_b, rng, args.lh_max_labels, args.lh_epsilon_tol, deadline
        )

    initial_p, initial_q = genetic_algorithm_initialization(
        payoff_a, payoff_b, rng, args.ga_population, args.ga_generations, args.ga_mutation, deadline
    )
    if args.init_method == "ga_only":
        metrics = compute_regrets(initial_p, initial_q, payoff_a, payoff_b)
        return {
            "p": initial_p,
            "q": initial_q,
            "metrics": metrics,
            "objective": metrics["R1"] + metrics["R2"],
            "success": False,  # No SLSQP solver was run in GA-only mode.
            "iterations": 0,
            "restart": 1,
            "refinement_applied": False,
            "timed_out": time_limit_reached(deadline),
        }

    result = refine_general_sum(initial_p, initial_q, payoff_a, payoff_b, args.maxiter, args.ftol, deadline)
    result["restart"] = 1
    result["refinement_applied"] = True
    return result



def load_payoff_dataset(input_path):
    """Load a dataset containing payoff_A and payoff_B with shape (K, M, N)."""
    input_path = Path(input_path)
    if not input_path.is_file():
        raise FileNotFoundError(f"Input dataset not found: {input_path}")

    with np.load(input_path) as dataset:
        if "payoff_A" not in dataset.files or "payoff_B" not in dataset.files:
            raise KeyError("Input .npz must contain arrays named 'payoff_A' and 'payoff_B'.")
        payoff_a = np.asarray(dataset["payoff_A"], dtype=np.float64)
        payoff_b = np.asarray(dataset["payoff_B"], dtype=np.float64)

    if payoff_a.ndim != 3 or payoff_b.ndim != 3 or payoff_a.shape != payoff_b.shape:
        raise ValueError("payoff_A and payoff_B must have the same three-dimensional shape (K, M, N).")
    return payoff_a, payoff_b


def save_results_to_excel(output_path, strategies_p, strategies_q, r1_values, r2_values,
                          eps_values, objectives, successes, restarts, solve_seconds,
                          refinement_applied, timed_out, approximate_success):
    """Save one row of refinement results per game to an Excel workbook."""
    missing = [name for name in ("pandas", "openpyxl") if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

    import pandas as pd

    num_games = len(eps_values)
    table = {
        "game_id": np.arange(num_games),
        "R1": np.asarray(r1_values),
        "R2": np.asarray(r2_values),
        "epsilon": np.asarray(eps_values),
        "objective_R1_plus_R2": np.asarray(objectives),
        "solver_success": np.asarray(successes),
        "approximate_nash_success": np.asarray(approximate_success),
        "best_restart": np.asarray(restarts),
        "runtime_seconds": np.asarray(solve_seconds),
        "refinement_applied": np.asarray(refinement_applied),
        "timed_out": np.asarray(timed_out),
    }
    for action_id in range(strategies_p.shape[1]):
        table[f"p_{action_id + 1}"] = strategies_p[:, action_id]
    for action_id in range(strategies_q.shape[1]):
        table[f"q_{action_id + 1}"] = strategies_q[:, action_id]

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        pd.DataFrame(table).to_excel(writer, sheet_name="Refinement_Results", index=False)


def main(args):
    if args.num_restarts < 1:
        raise ValueError("num-restarts must be at least 1.")
    if args.init_method in {"ga", "ga_only"} and args.ga_population < 3:
        raise ValueError("ga-population must be at least 3 for tournament selection.")
    if args.data_source == "generate" and args.num_games < 1:
        raise ValueError("num-games must be at least 1 when generating games.")

    rng = np.random.default_rng(args.seed)
    input_payoff_a = input_payoff_b = None
    if args.data_source == "file":
        if args.input is None:
            raise ValueError("--input PATH is required when --data-source file is selected.")
        input_payoff_a, input_payoff_b = load_payoff_dataset(args.input)
        available_games = input_payoff_a.shape[0]
        num_games = available_games if args.num_games == -1 else args.num_games
        if num_games < 1 or num_games > available_games:
            raise ValueError(f"num-games must be from 1 to {available_games}, or -1 to use all games.")
    else:
        num_games = args.num_games

    all_a, all_b, all_p, all_q = [], [], [], []
    r1_values, r2_values, eps_values = [], [], []
    objectives, successes, restarts = [], [], []
    refinement_flags, timed_out_flags = [], []
    solve_seconds = []
    total_start_time = time.perf_counter()

    for game_id in range(num_games):
        if args.data_source == "generate":
            payoff_a, payoff_b = generate_general_sum_game(
                args.m, args.n, args.distribution, args.mean, args.std,
                args.low, args.high, rng,
            )
        else:
            payoff_a, payoff_b = input_payoff_a[game_id], input_payoff_b[game_id]
        game_start_time = time.perf_counter()
        result = solve_with_initialization(payoff_a, payoff_b, args, rng)
        elapsed_seconds = time.perf_counter() - game_start_time
        metrics = result["metrics"]

        all_a.append(payoff_a); all_b.append(payoff_b)
        all_p.append(result["p"]); all_q.append(result["q"])
        r1_values.append(metrics["R1"]); r2_values.append(metrics["R2"])
        eps_values.append(metrics["epsilon"]); objectives.append(result["objective"])
        successes.append(result["success"]); restarts.append(result["restart"])
        solve_seconds.append(elapsed_seconds)
        refinement_flags.append(result["refinement_applied"])
        timed_out_flags.append(result.get("timed_out", False))

        print(
            f"Game {game_id + 1:>{len(str(num_games))}}/{num_games} | "
            f"R1={metrics['R1']:.3e} | R2={metrics['R2']:.3e} | "
            f"epsilon={metrics['epsilon']:.3e} | time={elapsed_seconds:.3f}s"
            f"{' | TIME LIMIT' if result.get('timed_out', False) else ''}"
        )

    eps_values = np.asarray(eps_values)
    solve_seconds = np.asarray(solve_seconds)
    approximate_success = eps_values <= args.success_epsilon
    timed_out_values = np.asarray(timed_out_flags, dtype=bool)
    total_elapsed_seconds = time.perf_counter() - total_start_time
    output_path = Path(args.output)
    strategies_p = np.stack(all_p)
    strategies_q = np.stack(all_q)
    np.savez_compressed(
        output_path,
        payoff_A=np.stack(all_a), payoff_B=np.stack(all_b),
        strategy_p=strategies_p, strategy_q=strategies_q,
        R1=np.asarray(r1_values), R2=np.asarray(r2_values), epsilon=eps_values,
        objective=np.asarray(objectives), solver_success=np.asarray(successes),
        approximate_nash_success=approximate_success,
        success_epsilon=np.asarray(args.success_epsilon),
        best_restart=np.asarray(restarts), runtime_seconds=solve_seconds,
        initialization_method=np.asarray(args.init_method),
        data_source=np.asarray(args.data_source),
        refinement_applied=np.asarray(refinement_flags),
        timed_out=np.asarray(timed_out_flags),
    )

    if args.excel_output:
        save_results_to_excel(
            args.excel_output, strategies_p, strategies_q, r1_values, r2_values,
            eps_values, objectives, successes, restarts, solve_seconds, refinement_flags, timed_out_flags,
            approximate_success,
        )

    print("\n" + "=" * 64)
    print(f"Saved: {output_path.resolve()}")
    if args.excel_output:
        print(f"Excel summary saved: {Path(args.excel_output).resolve()}")
    print(f"Games: {num_games} | Game size: {all_a[0].shape[0]}x{all_a[0].shape[1]} | Initialization: {args.init_method}")
    print(f"Data source: {args.data_source}")
    print(f"Time limit/game: {'unlimited' if args.time_limit <= 0 else f'{args.time_limit:g} s'}")
    if args.init_method == "random":
        print(f"Standard random restarts/game: {args.num_restarts}")
    elif args.init_method == "informed_random":
        print(f"Payoff-informed random restarts/game: {args.num_restarts}")
    elif args.init_method in {"ga", "ga_only"}:
        print(f"GA population: {args.ga_population} | GA generations: {args.ga_generations}")
    elif args.init_method == "lemke_howson":
        print(f"Lemke--Howson labels/game: {args.lh_max_labels} | epsilon tolerance: {args.lh_epsilon_tol:g}")
    else:
        print(f"Smooth-epsilon steps: {args.smooth_steps} | learning rate: {args.smooth_lr}")
    print(f"Mean epsilon: {eps_values.mean():.6e}")
    print(f"Median epsilon: {np.median(eps_values):.6e}")
    print(
        f"Success rate (epsilon <= {args.success_epsilon:g}): "
        f"{100 * approximate_success.mean():.2f}%"
    )
    print(
        f"Timeouts: {timed_out_values.sum()}/{num_games} "
        f"({100 * timed_out_values.mean():.2f}%)"
    )
    print(f"Mean solve time/game: {solve_seconds.mean():.3f} s")
    print(f"Median solve time/game: {np.median(solve_seconds):.3f} s")
    print(f"Total elapsed time: {total_elapsed_seconds:.3f} s")
    for threshold in (1e-6, 1e-4, 1e-3, 1e-2, 0.05, 0.10, 0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.5,2.0):
        print(f"epsilon <= {threshold:g}: {100 * np.mean(eps_values <= threshold):.2f}%")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="General-sum game refinement dataset generator")
    parser.add_argument("--num-games", type=int, default=100, help="Games to process; use -1 for all input-file games")
    parser.add_argument("--data-source", choices=["generate", "file"], default="generate")
    parser.add_argument("--input", default=None, help="Input .npz with payoff_A and payoff_B; required with --data-source file")
    parser.add_argument("--num-restarts", type=int, default=10, help="Number of random or informed-random SLSQP starts per game")
    parser.add_argument("--init-method",choices=["random", "informed_random", "smooth_only", "smooth_norm_only","smooth_slsqp", "ga", "ga_only", "lemke_howson"], default="random",help="smooth_norm_only is an alias for smooth_only",)
    parser.add_argument("--ga-population", type=int, default=100)
    parser.add_argument("--ga-generations", type=int, default=100)
    parser.add_argument("--ga-mutation", type=float, default=0.05)
    parser.add_argument("--smooth-steps", type=int, default=1000)
    parser.add_argument("--smooth-lr", type=float, default=0.05)
    parser.add_argument("--smooth-device", default=None, help="Set to cuda or cpu; default selects CUDA when available")
    parser.add_argument("--lh-max-labels", type=int, default=1, help="Number of dropped labels to try; use 0 for all labels")
    parser.add_argument("--lh-epsilon-tol", type=float, default=1e-6)
    parser.add_argument("--m", type=int, default=5, help="Player 1 action count")
    parser.add_argument("--n", type=int, default=5, help="Player 2 action count")
    parser.add_argument("--distribution", choices=["normal", "uniform"], default="normal")
    parser.add_argument("--mean", type=float, default=50.0)
    parser.add_argument("--std", type=float, default=5.0, help="Normal-distribution standard deviation")
    parser.add_argument("--low", type=float, default=10.0)
    parser.add_argument("--high", type=float, default=90.0)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--maxiter", type=int, default=5000)
    parser.add_argument("--ftol", type=float, default=1e-12)
    parser.add_argument("--time-limit", type=float, default=0.0,
                        help="Maximum wall-clock seconds per game; 0 disables the limit")
    parser.add_argument("--success-epsilon", type=float, default=0.10,
                        help="A game is successful when its exact epsilon is at most this value")
    parser.add_argument("--output", default="general_sum_refined_dataset.npz")
    parser.add_argument("--excel-output", default=None, help="Optional .xlsx path for one-row-per-game results")

    M = 20
    N = 20
    METHOD = "lemke_howson"  #choices=["random", "smooth_slsqp", "smooth_only","ga", "ga_only", "lemke_howson"]
    DISTRIBUTION = "uniform"  # "normal" or "uniform"
    BASE_SEED = 2026

    INPUT_FILE = (f"input_games_{DISTRIBUTION}_{M}_{N}.npz")
    RESULT_FILE = (f"{METHOD}_{DISTRIBUTION}_{M}_{N}.xlsx")

    main(parser.parse_args([
        "--data-source", "file",
        "--input", INPUT_FILE,
        "--num-games", "-1",
        "--time-limit", "180",      # 0 = unlimit
        "--success-epsilon", "0.5",
        "--seed", "2026",
        "--init-method", METHOD,
        "--excel-output", RESULT_FILE,

        # Random+SLSQP
        "--num-restarts", "1",
        "--maxiter", "5000",
        "--ftol", "1e-12",

        # GA-only
        "--ga-population", "500",
        "--ga-generations", "1000",
        "--ga-mutation", "0.05",
        # GA+SLSQP
        "--ga-population", "150",
        "--ga-generations", "1000",
        "--ga-mutation", "0.05",
        "--maxiter", "5000",
        "--ftol", "1e-12",

        # Lemke--Howson
        "--lh-max-labels", "1",   #0
        "--lh-epsilon-tol", "1e-6",

        # Smooth-epsilon+SLSQP & Smooth-epsilon only
        "--smooth-steps", "1000",
        "--smooth-lr", "0.01",
        "--maxiter", "5000",
        "--ftol", "1e-10",
        "--smooth-device", "cpu",

    ]))

    GOOGLEDRIVE = "/content/drive/MyDrive/Research/SLSQP/results"
    excel_FILE_GOOGLE = f"{GOOGLEDRIVE}/{RESULT_FILE}"

    shutil.copyfile(
        RESULT_FILE,
        excel_FILE_GOOGLE
    )
    print(f"Copied file      : {RESULT_FILE}")








Game   1/150 | R1=1.460e+01 | R2=1.113e+01 | epsilon=1.460e+01 | time=180.035s | TIME LIMIT
Game   2/150 | R1=0.000e+00 | R2=0.000e+00 | epsilon=0.000e+00 | time=0.023s
Game   3/150 | R1=1.117e+01 | R2=8.991e+00 | epsilon=1.117e+01 | time=180.104s | TIME LIMIT
Game   4/150 | R1=0.000e+00 | R2=0.000e+00 | epsilon=0.000e+00 | time=0.012s
Game   5/150 | R1=0.000e+00 | R2=0.000e+00 | epsilon=0.000e+00 | time=0.011s
Game   6/150 | R1=1.421e-14 | R2=0.000e+00 | epsilon=1.421e-14 | time=0.010s
Game   7/150 | R1=0.000e+00 | R2=0.000e+00 | epsilon=0.000e+00 | time=0.011s
Game   8/150 | R1=1.171e+01 | R2=9.161e+00 | epsilon=1.171e+01 | time=0.012s
Game   9/150 | R1=1.297e+01 | R2=1.386e+01 | epsilon=1.386e+01 | time=180.104s | TIME LIMIT
Game  10/150 | R1=1.435e+01 | R2=1.189e+01 | epsilon=1.435e+01 | time=180.104s | TIME LIMIT
Game  11/150 | R1=1.072e+01 | R2=2.095e+01 | epsilon=2.095e+01 | time=180.103s | TIME LIMIT
Game  12/150 | R1=0.000e+00 | R2=0.000e+00 | epsilon=0.000e+00 | time=0.010s
G